In [ ]:
#מטרת הפרוקט היא לנתח את הביצועים של השחקנים בטורנירים 
#שאלות יהיו מי השחקנים עם הזכיות הכי גבוהות ? איזה שנים היו הכו רןןחיות? מאיזו מדינה באים השחקנים הכי מורווחים?
# כלים שאני אשתמש הם pandas לנתוח ניתונים ; seaborn, matplotlib לוויזואליזציה

import pandas as pd

In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=-PC\\SQLEXPRESS;"
    "DATABASE=POKER;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;")

In [ ]:
Player = pd.read_sql('select * from Player' , conn)
Results = pd.read_sql('select * from results' , conn)
Tournament = pd.read_sql('select * from tournament' , conn)


In [ ]:
Player.info()

In [ ]:
Results.info()

In [ ]:
Tournament.info()

In [ ]:
Player.head()

In [ ]:
Results.head()

In [ ]:
Tournament.head()

In [ ]:
df = Player.merge(Results , left_on = 'ID'  , right_on =  'Player_Id').merge(Tournament , left_on = 'Tournament_Id' , right_on = 'ID')

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=["ID_x", "ID_y", "ID"])

In [ ]:
df.head()  
# כאן רואים את מבנה הנתונים:
# שם השחקן, אזרחות, מספר צמידים,
# סך הזכיות בקריירה,
# המיקום בטורניר מסוים,
# סכום הזכייה בטורניר,
# המדינה שבה התקיים הטורניר,
# השנה ושם הטורניר

In [ ]:
df.info()

In [ ]:
df.describe()

#  מהסטטיסטיקות ניתן לראות כמה דברים מעניינים
# רוב הזכיות בטורנירים אינן גבוהות מאוד אבל יש מספר קטן של זכיות ענק שמעלות את הממוצע
# ניתן לראות פער גדול בין הממוצע לבין החציון בעמודת הזכיות
# הנתונים כוללים טורנירים משנת 1971 עד 2025 ולכן אפשר לנתח מגמות לאורך שנים
# מספר השחקנים בטורנירים משתנה מאוד יש טורנירים קטנים ויש טורנירים עם אלפי משתתפים
# דמי הכניסה לטורנירים דומים יחסית ונעים בין 5000 ל 10000

In [ ]:
df.isnull().sum() 
# לא חסרים ערכים אין שורות למחוק
# אין צורך בלהחליף את ה 'edition_year' ל datetime כי זה לא תאריך אלה רק מספר שנה

In [ ]:
import matplotlib.pyplot as plt

top_players = df.groupby('Name')['Win'].sum().sort_values(ascending=False).head(10) / 1000000

top_players.plot(kind='barh')

plt.ylabel('Name')
plt.xlabel('Winnings in millions usd')
plt.title('Top 10 players by total winnings')
plt.show()
#ניתן כאן לראות את ה10 שחקנים שהרוויחו הכי הרבה כסף במיליוני דולרים 
#הניתוח לפי שנים מבוסס על הנתונים הקיימים במסד הנתונים. יש שנים שבהן קיימות מעט רשומות ולכן הסכום מייצג רק חלק מהתוצאות 

In [ ]:
wins_by_year = df.groupby('Edition_Year')['Win'].sum()/1000000


wins_by_year.plot(kind='line', title='Total winnings by year in millions usd')
#אפשר לראות פה את ההתקדמות עם הזמן של הסכומים שמורווחים על ידי השחקנים
#הניתוח לפי שנים מבוסס על הנתונים הקיימים במסד הנתונים. יש שנים שבהן קיימות מעט רשומות ולכן הסכום מייצג רק חלק מהתוצאות 

In [ ]:
top_countries = df.groupby('Nationality')['Win'].sum().sort_values(ascending=False).head(10)/1000000

top_countries.plot(kind='bar', title='Top countries by total winnings')
#אפשר לראות פה את המדינות אשר באי השחקנים ם הכי הרבה זכיות
#הניתוח לפי שנים מבוסס על הנתונים הקיימים במסד הנתונים. יש שנים שבהן קיימות מעט רשומות ולכן הסכום מייצג רק חלק מהתוצאות 

In [ ]:
import seaborn as sns

data = df[['Position','Win','Prize_Pool']]

sns.heatmap(data.corr(), annot=True, cmap='coolwarm')

plt.title('Correlation between position winnings and prize pool')
plt.show()
# קיים קשר שלילי חלש בין המיקום בטורניר לבין סכום הזכייה כלומר ככל שהמיקום טוב יותר הזכייה בדרך כלל גבוהה יותר
#  קיים קשר חיובי בין הפיזפול לבן מספר השחקנים שמרויחים כסף   
# אין קשר בין סכום הפרייזפול לבין הרווח ש השחקן
#הניתוח לפי שנים מבוסס על הנתונים הקיימים במסד הנתונים. יש שנים שבהן קיימות מעט רשומות ולכן הסכום מייצג רק חלק מהתוצאות 

In [ ]:
df.plot(kind='scatter', x='Num_Of_Players', y='Prize_Pool',
        title='Number of players vs Prize pool')
#אפשר לראות פה שכל עוד המספר של השחקנים גדל הזכייה תהיה יות גדולה
#הניתוח לפי שנים מבוסס על הנתונים הקיימים במסד הנתונים. יש שנים שבהן קיימות מעט רשומות ולכן הסכום מייצג רק חלק מהתוצאות 

In [ ]:
# Conclusion
# מהניתוח ניתן לראות כי מספר קטן של שחקנים זוכה ברוב הכסף
# ארצות הברית מובילה במספר הזכיות בטורנירים
# ככל שמספר השחקנים בטורניר גדול יותר כך גובה סכום הפרסים בטורניר גדל
# בנוסף ניתן לראות קשר בין המיקום בטורניר לבין סכום הזכייה